# 🚀 GIAI ĐOẠN 3 — HUẤN LUYỆN MÔ HÌNH STAIR-v5 (STAIR-BSC-Reweight)
### *Safe Topological Reweighting with Multiplicative Consensus Boosting and SPSD-Guaranteed BSC Smoother*
**Đề tài Khóa Luận Tốt Nghiệp — Khoa CNTT, Trường ĐH Khoa học Tự nhiên, ĐHQG-HCM**

---

## 📌 1. BỐI CẢNH & KHẮC PHỤC 5 TỬ HUYỆT KỸ THUẬT

Phiên bản **STAIR-v5 (STAIR-BSC-Reweight)** được xây dựng trên cơ sở chuẩn tắc của **STAIR (AAAI 2025)**, khắc phục triệt để 5 tử huyệt toán học và môi trường đã được mổ xẻ trong báo cáo thẩm định:

1. **100% SPSD Guaranteed (Đối xứng hóa & Chuẩn hóa Laplacian)**:
   $$W_{\text{sym}} = \max(W, W^T), \quad \tilde{S} = D^{-1/2} W_{\text{sym}} D^{-1/2}$$
   Bảo đảm bán kính phổ $\rho(\tilde{S}) \le 1.0$ và toàn bộ trị riêng $\ge -1.0$. Triệt tiêu hoàn toàn nguy cơ đảo dấu gradient trong bộ làm mịn BSC Smoother.

2. **0% Edge Pruning (Bảo tồn 100% Tô-pô Đồ thị Baseline)**:
   Giữ nguyên toàn bộ liên kết kNN gốc ($k_{\text{text}}=5, k_{\text{vis}}=1$, $40,459$ cạnh trên Baby, $>110,000$ cạnh trên Sports). Bậc đỉnh trung bình giữ nguyên $6 - 8$, loại bỏ hoàn toàn nguy cơ đói cấu trúc và bảo vệ $100\%$ sản phẩm đuôi dài.

3. **Multiplicative Consensus Boost (Bảo toàn Đơn điệu Trọng số Đồng thuận)**:
   $$W_{ij} = W_{\text{base}} \cdot (1 + \alpha \cdot q_{ij}^{\text{modal}} + \beta \cdot q_{ij}^{\text{behavior}}), \quad W_{ij} \in [1.00, 3.60]$$
   Với $W_{\text{base}} \in \{1.0, 2.0\}$, cạnh đồng thuận kép luôn có trọng số vượt trội cạnh đơn phương thức ($W^{(2.0)} \ge 2 \times W^{(1.0)}$).

4. **BPR Loss Liên tục (Bảo toàn Gradient Dày đặc 100%)**:
   Duy trì BPR Loss mịn $\mathcal{L}_{\text{BPR}} = -\ln \sigma(\hat{y}_{ui} - \hat{y}_{uj})$, duy trì gradient dày đặc cung cấp năng lượng đầy đủ cho toán tử làm mịn BSC: $\Delta W = \sum_{l=0}^L \beta_l \tilde{S}^l G$.

5. **SVD Whitening Chuẩn tắc Đại số Tuyến tính**:
   Phân rã SVD $X_c = U S V^T \implies E = U_{[:, :d]} \times \sqrt{N / d}$. Triệt tiêu hoàn toàn singular values $S$, đưa ma trận hiệp phương sai về dạng cầu đẳng hướng $\text{Cov} = \frac{1}{d} I_d$.

---

## 🎯 2. BẢNG ĐỐI CHỨNG BASELINE CHUẨN MỰC (Theo Table 3.1 `03_stair.tex`)

| Tập Dữ Liệu | Chỉ Số | STAIR Baseline (03_stair.tex) | v4.1 Thực Đo | STAIR-v5 Kỳ Vọng | Mục Tiêu Khoa Học |
| :--- | :--- | :---: | :---: | :---: | :--- |
| **Amazon Sports** | **Recall@20** | **0.1111** | 0.1116 (+0.45%) | **0.1120 — 0.1130** | Bứt phá hiệu năng đồ thị mật độ TB |
| *(Mật độ 0.05%)* | **NDCG@20** | **0.0500** | 0.0502 (+0.40%) | **0.0505 — 0.0515** | Bứt phá toàn diện NDCG |
| **Amazon Baby** | **Recall@20** | **0.1042** | 0.0947 (-9.12%) | **0.1045 — 0.1055** | Khắc phục suy giảm trên đồ thị siêu thưa |
| *(Siêu thưa)* | **NDCG@20** | **0.0454** | 0.0415 (-8.59%) | **0.0455 — 0.0465** | Phục hồi và vượt baseline |
| **Amazon Electronics** | **Recall@20** | **0.0665** | Chưa chạy | **0.0675 — 0.0685** | Mở rộng quy mô lớn 63K items |
| *(Quy mô 63K)* | **NDCG@20** | **0.0303** | Chưa chạy | **0.0308 — 0.0315** | An toàn 0 MB dense overhead |


In [ ]:
# Cell 2: Kiểm tra Môi trường & Thiết bị GPU
!nvidia-smi
!python --version
import torch

print("=" * 65)
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_bytes = torch.cuda.get_device_properties(0).total_memory
    print(f"GPU Model       : {device_name}")
    print(f"VRAM Capacity   : {vram_bytes / 1024**3:.2f} GB ({vram_bytes / 1024**2:.0f} MB)")
else:
    print("⚠️ CẢNH BÁO: CUDA không khả dụng. Tiến trình sẽ chạy trên CPU (rất chậm)!")
print("=" * 65)


In [ ]:
# Cell 3: Đồng bộ STAIR-Enhanced (v5) & Cài đặt Dependencies
import os, shutil, subprocess, sys

STAIR_DIR = '/kaggle/working/STAIR-Enhanced'
os.chdir('/kaggle/working')

# 1. Clone hoặc đồng bộ cưỡng bức repository từ origin/main
if os.path.exists(STAIR_DIR):
    print("Thư mục STAIR-Enhanced đã tồn tại. Đang đồng bộ cưỡng bức commit mới nhất...")
    try:
        subprocess.run(['git', '-C', STAIR_DIR, 'fetch', 'origin', 'main'], check=True)
        subprocess.run(['git', '-C', STAIR_DIR, 'reset', '--hard', 'origin/main'], check=True)
        print("✅ Đã reset về commit mới nhất của origin/main thành công.")
    except Exception as e:
        print(f"Lỗi git fetch/reset ({e}), đang clone lại từ đầu...")
        shutil.rmtree(STAIR_DIR, ignore_errors=True)

if not os.path.exists(STAIR_DIR):
    print("Cloning STAIR-Enhanced repository (branch main)...")
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/ThanhChuong12/STAIR-Enhanced.git', STAIR_DIR
    ], check=True)

for p in [STAIR_DIR, '/kaggle/working']:
    if p not in sys.path:
        sys.path.insert(0, p)

if os.path.exists(STAIR_DIR):
    os.chdir(STAIR_DIR)

# Xóa cache module để kernel luôn nạp phiên bản v5 mới nhất
for mod_name in list(sys.modules.keys()):
    if 'stair_sre_v5' in mod_name or 'models.stair_sre_v5' in mod_name:
        sys.modules.pop(mod_name, None)

# 2. Cài đặt các gói phụ thuộc bắt buộc
print("📦 Cài đặt dependencies (torchdata, freerec, nvidia-ml-py, prettytable, pyyaml)...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'torchdata==0.7.1'], check=False)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'freerec==0.8.5', 'nvidia-ml-py', 'prettytable', 'matplotlib', 'pyyaml', 'seaborn', 'pandas', 'scipy', 'pytest'
], check=True)

# 2b. Cai dat torch-geometric (bat buoc cho freerec.graph)
import torch as _torch_tmp
_TORCH_VER = _torch_tmp.__version__.split('+')[0]
_CUDA_TAG  = 'cu' + _torch_tmp.version.cuda.replace('.','') if _torch_tmp.cuda.is_available() else 'cpu'
print(f"Cai dat torch-geometric (torch={_TORCH_VER}, {_CUDA_TAG})...")
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', 'torch-geometric',
    '-f', f'https://data.pyg.org/whl/torch-{_TORCH_VER}+{_CUDA_TAG}.html'
], check=False)

try:
    import torch_geometric
    print(f"torch-geometric {torch_geometric.__version__} da san sang.")
except ImportError:
    print("Cai dat torch-geometric tu PyPI (fallback)...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-geometric'], check=False)
    try:
        import torch_geometric
        print("torch-geometric (PyPI) da san sang.")
    except ImportError:
        print("CANH BAO: Khong the cai dat torch-geometric! freerec.graph se that bai.")


# 3. Kaggle TorchData compatibility shims & Idempotent DataPipe Registration
import types
import torch.utils.data

try:
    from torch.utils.data.datapipes.datapipe import IterDataPipe as _NativeIterDP, MapDataPipe as _NativeMapDP
    for _cls in [_NativeIterDP, _NativeMapDP]:
        if hasattr(_cls, 'register_datapipe_as_function'):
            _orig_reg = _cls.register_datapipe_as_function
            def _make_safe_reg(orig_fn):
                def _safe_reg(cls, function_name, cls_to_register, *args, **kwargs):
                    if hasattr(cls, 'functions') and function_name in cls.functions:
                        try:
                            del cls.functions[function_name]
                        except Exception:
                            pass
                    return orig_fn.__func__(cls, function_name, cls_to_register, *args, **kwargs)
                return _safe_reg
            _cls.register_datapipe_as_function = classmethod(_make_safe_reg(_orig_reg))
except Exception:
    pass

try:
    import torchdata
    import torchdata.datapipes as dp
except Exception:
    dp = None

if dp is None or 'torchdata.datapipes' not in sys.modules:
    if 'torchdata' not in sys.modules:
        td = types.ModuleType('torchdata')
        sys.modules['torchdata'] = td
    else:
        td = sys.modules['torchdata']
    dp = types.ModuleType('torchdata.datapipes')
    td.datapipes = dp
    sys.modules['torchdata.datapipes'] = dp

if not hasattr(dp, 'iter'):
    iter_mod = types.ModuleType('torchdata.datapipes.iter')
    dp.iter = iter_mod
    sys.modules['torchdata.datapipes.iter'] = iter_mod
if not hasattr(dp.iter, 'IterDataPipe'):
    class IterDataPipe(torch.utils.data.IterableDataset):
        def __iter__(self): return iter([])
    dp.iter.IterDataPipe = IterDataPipe

if not hasattr(dp, 'map'):
    map_mod = types.ModuleType('torchdata.datapipes.map')
    dp.map = map_mod
    sys.modules['torchdata.datapipes.map'] = map_mod
if not hasattr(dp.map, 'MapDataPipe'):
    class MapDataPipe(torch.utils.data.Dataset):
        def __getitem__(self, idx): raise NotImplementedError
        def __len__(self): return 0
    dp.map.MapDataPipe = MapDataPipe

# 4. Xác nhận sự hiện diện của các file STAIR-v5 bắt buộc
v5_model_path = os.path.join(STAIR_DIR, 'models', 'stair_sre_v5.py')
v5_main_path  = os.path.join(STAIR_DIR, 'mainS3_v5.py')
v5_check_path = os.path.join(STAIR_DIR, 'tests', 'static_sanity_check_v5.py')

assert os.path.exists(v5_model_path), f"LỖI: Không tìm thấy {v5_model_path}!"
assert os.path.exists(v5_main_path),  f"LỖI: Không tìm thấy {v5_main_path}!"
assert os.path.exists(v5_check_path), f"LỖI: Không tìm thấy {v5_check_path}!"

print("=" * 80)
print(f"✅ Model v5 Path         : {v5_model_path}")
print(f"✅ Main v5 Runner Path   : {v5_main_path}")
print(f"✅ Static Check Script   : {v5_check_path}")
print("=" * 80)


In [ ]:
# Cell 4: Chuẩn bị Dữ liệu & Cầu nối Đa Vị trí (Data Preparation & Multi-Bridge)
import os, shutil, glob, zipfile, tarfile

STAIR_DIR = '/kaggle/working/STAIR-Enhanced'
DATA_ROOT = '/kaggle/data'
PROCESSED_ROOT = os.path.join(DATA_ROOT, 'Processed')
LOCAL_DATA = os.path.join(STAIR_DIR, 'data')
LOCAL_PROCESSED = os.path.join(LOCAL_DATA, 'Processed')

for d in [DATA_ROOT, PROCESSED_ROOT, LOCAL_DATA, LOCAL_PROCESSED]:
    os.makedirs(d, exist_ok=True)

DATASET_NAMES = [
    'Amazon2014Sports_550_MMRec',
    'Amazon2014Baby_550_MMRec',
    'Amazon2014Electronics_550_MMRec'
]

DATASET_ALIASES = {
    'Amazon2014Sports_550_MMRec': ['sport', 'sports', 'amazon2014sports'],
    'Amazon2014Baby_550_MMRec': ['baby', 'amazon2014baby'],
    'Amazon2014Electronics_550_MMRec': ['electronic', 'electronics', 'amazon2014electronics']
}

REQUIRED_FILES = {'train.txt', 'valid.txt', 'test.txt', 'textual_modality.pkl', 'visual_modality.pkl'}

def bridge_directories(src_dir, target_folder):
    destinations = [
        os.path.join(DATA_ROOT, target_folder),
        os.path.join(PROCESSED_ROOT, target_folder),
        os.path.join(LOCAL_DATA, target_folder),
        os.path.join(LOCAL_PROCESSED, target_folder),
    ]
    for dst in destinations:
        if os.path.abspath(src_dir) == os.path.abspath(dst):
            continue
        os.makedirs(dst, exist_ok=True)
        for item in os.listdir(src_dir):
            s_item = os.path.join(src_dir, item)
            d_item = os.path.join(dst, item)
            if os.path.isfile(s_item) and not os.path.exists(d_item):
                try:
                    os.symlink(s_item, d_item)
                except Exception:
                    shutil.copy2(s_item, d_item)

prepared_datasets = {}
input_base = '/kaggle/input'

print("🔍 Đang quét và đồng bộ dữ liệu vào hệ thống FreeRec...")
for ds_name in DATASET_NAMES:
    keywords = DATASET_ALIASES.get(ds_name, [ds_name.lower()])
    found_dir = None

    # 1. Kiểm tra nếu đã có sẵn tại bất kỳ destination nào
    for cand_dir in [
        os.path.join(PROCESSED_ROOT, ds_name),
        os.path.join(DATA_ROOT, ds_name),
        os.path.join(LOCAL_PROCESSED, ds_name),
        os.path.join(LOCAL_DATA, ds_name),
    ]:
        if os.path.exists(cand_dir) and len(os.listdir(cand_dir)) >= 5:
            if REQUIRED_FILES.issubset(set(os.listdir(cand_dir))):
                found_dir = cand_dir
                break

    # 2. Quét trong /kaggle/input
    if found_dir is None and os.path.exists(input_base):
        for root, dirs, files in os.walk(input_base):
            if ds_name in dirs:
                cand = os.path.join(root, ds_name)
                if REQUIRED_FILES.issubset(set(os.listdir(cand))):
                    found_dir = cand
                    break
        if found_dir is None:
            for root, dirs, files in os.walk(input_base):
                has_modals = any('modality.pkl' in f for f in files)
                has_txt = any(f in files for f in ['train.txt', 'valid.txt', 'test.txt'])
                dir_lower = root.lower()
                if (has_modals and has_txt) and any(kw in dir_lower for kw in keywords):
                    found_dir = root
                    break

    # 3. Quét file nén nếu có
    if found_dir is None:
        for search_root in [input_base, DATA_ROOT, '/kaggle/working']:
            if os.path.exists(search_root):
                for r, _, fnames in os.walk(search_root):
                    for fn in fnames:
                        if fn.endswith(('.zip', '.tar.gz', '.tar', '.tgz')) and any(kw in fn.lower() for kw in keywords):
                            arc_path = os.path.join(r, fn)
                            dst_extract = os.path.join(PROCESSED_ROOT, ds_name)
                            os.makedirs(dst_extract, exist_ok=True)
                            print(f"  [Giải nén] {arc_path} -> {dst_extract}...")
                            if fn.endswith('.zip'):
                                with zipfile.ZipFile(arc_path, 'r') as zf:
                                    zf.extractall(dst_extract)
                            else:
                                with tarfile.open(arc_path, 'r:*') as tf:
                                    tf.extractall(dst_extract)
                            subitems = os.listdir(dst_extract)
                            if len(subitems) == 1 and os.path.isdir(os.path.join(dst_extract, subitems[0])):
                                nested = os.path.join(dst_extract, subitems[0])
                                for nf in os.listdir(nested):
                                    shutil.move(os.path.join(nested, nf), os.path.join(dst_extract, nf))
                                os.rmdir(nested)
                            found_dir = dst_extract
                            break
                    if found_dir is not None: break
            if found_dir is not None: break

    if found_dir is not None:
        bridge_directories(found_dir, ds_name)
        target_check = os.path.join(LOCAL_PROCESSED, ds_name)
        present = set(os.listdir(target_check))
        missing = REQUIRED_FILES - present
        if missing:
            print(f"⚠️ [{ds_name}] Thiếu file: {missing}")
        else:
            print(f"✅ [{ds_name}] Sẵn sàng 5 file dữ liệu bắt buộc.")
            prepared_datasets[ds_name] = target_check
    else:
        print(f"ℹ️ [{ds_name}] Chưa tìm thấy trong /kaggle/input.")

print("=" * 80)
print(f"TỔNG KẾT: {len(prepared_datasets)} / {len(DATASET_NAMES)} tập dữ liệu sẵn sàng trong FreeRec Processed/")
print("=" * 80)


In [ ]:
# Cell 5: Bước 0 — Chạy Unit Tests Toán Học & Static Sanity Check (0s GPU)
import os, sys

os.chdir(STAIR_DIR)

# 1. Chạy 5/5 PyTest Unit Tests (Đại số tuyến tính & Toán lý thuyết)
print("🧪 [Kiểm thử 1/2] Đang thực thi Unit Tests STAIR-v5 (python tests/test_stair_v5.py)...\n")
!python tests/test_stair_v5.py

# 2. Chạy Static Sanity Check trên dữ liệu thực tế
print("\n" + "=" * 85)
print("🔬 [Kiểm thử 2/2] Đang chạy Static Sanity Check (Bước 0) trên dữ liệu thực tế...")
print("=" * 85 + "\n")
!python tests/static_sanity_check_v5.py


In [ ]:
# Cell 6: Khởi tạo Training Engine v5, Hardware Profiler & Trích xuất Metrics
import os, sys, time, threading, subprocess, re
import torch
import pynvml
import pandas as pd

STAIR_DIR = '/kaggle/working/STAIR-Enhanced'
LOG_DIR_V5 = '/kaggle/working/logs/STAIR-v5'
os.makedirs(LOG_DIR_V5, exist_ok=True)

# Bảng tham chiếu Baseline chuẩn tắc (03_stair.tex Table 3.1) và v4.1 thực đo
BENCHMARK_TARGETS = {
    'Amazon2014Sports_550_MMRec': {
        'baseline': {'Recall@10': 0.0743, 'Recall@20': 0.1111, 'NDCG@10': 0.0405, 'NDCG@20': 0.0500},
        'v4_1_ssb': {'Recall@10': 0.0754, 'Recall@20': 0.1116, 'NDCG@10': 0.0409, 'NDCG@20': 0.0502},
        'target':   {'Recall@10': 0.0760, 'Recall@20': 0.1125, 'NDCG@10': 0.0420, 'NDCG@20': 0.0515},
    },
    'Amazon2014Baby_550_MMRec': {
        'baseline': {'Recall@10': 0.0674, 'Recall@20': 0.1042, 'NDCG@10': 0.0359, 'NDCG@20': 0.0454},
        'v4_1_ssb': {'Recall@10': 0.0620, 'Recall@20': 0.0947, 'NDCG@10': 0.0326, 'NDCG@20': 0.0415},
        'target':   {'Recall@10': 0.0680, 'Recall@20': 0.1050, 'NDCG@10': 0.0365, 'NDCG@20': 0.0460},
    },
    'Amazon2014Electronics_550_MMRec': {
        'baseline': {'Recall@10': 0.0442, 'Recall@20': 0.0665, 'NDCG@10': 0.0246, 'NDCG@20': 0.0303},
        'v4_1_ssb': {'Recall@10': 0.0000, 'Recall@20': 0.0000, 'NDCG@10': 0.0000, 'NDCG@20': 0.0000},
        'target':   {'Recall@10': 0.0450, 'Recall@20': 0.0680, 'NDCG@10': 0.0255, 'NDCG@20': 0.0315},
    },
}

vram_stats = {}

def vram_monitor(ds_key, stop_event, interval=1.0):
    """Theo dõi mức tiêu thụ VRAM GPU trong suốt quá trình huấn luyện."""
    try:
        pynvml.nvmlInit()
        handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        vram_stats[ds_key] = []
        while not stop_event.is_set():
            mem = pynvml.nvmlDeviceGetMemoryInfo(handle)
            vram_stats[ds_key].append(mem.used / (1024 * 1024))
            time.sleep(interval)
        pynvml.nvmlShutdown()
    except Exception:
        pass

def normalize_metric_name(k):
    parts = k.split('@')
    if len(parts) == 2:
        prefix = parts[0].upper()
        if prefix == 'RECALL':
            return f"Recall@{parts[1]}"
        elif prefix == 'NDCG':
            return f"NDCG@{parts[1]}"
    return k

def parse_best_metrics(log_path):
    """Trích xuất checkpoint tối ưu và metrics từ log file (chuẩn hóa tên metrics)."""
    if not os.path.exists(log_path):
        return None, {}
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()

    best_ep = None
    m_ep = re.search(r'Load best model @Epoch\s+(\d+)', content)
    if not m_ep:
        m_ep = re.search(r'Best @Epoch\s+(\d+)', content)
    if m_ep:
        best_ep = int(m_ep.group(1))

    metrics = {}
    test_matches = list(re.finditer(r'TEST\s+@Epoch:\s+\d+\s+>>>\s+\|\|\s*(.*?)\n', content))
    if test_matches:
        for m in re.finditer(r'([\w@]+)\s+Avg:\s+([\d.]+)', test_matches[-1].group(1)):
            k = m.group(1)
            v = float(m.group(2))
            metrics[k] = v
            metrics[normalize_metric_name(k)] = v
        return best_ep, metrics

    val_matches = list(re.finditer(r'VALID\s+@Epoch:\s+\d+\s+>>>\s+\|\|\s*(.*?)\n', content))
    if val_matches:
        for m in re.finditer(r'([\w@]+)\s+Avg:\s+([\d.]+)', val_matches[-1].group(1)):
            k = m.group(1)
            v = float(m.group(2))
            metrics[k] = v
            metrics[normalize_metric_name(k)] = v
        return best_ep, metrics

    return best_ep, metrics

def run_training_v5(
    dataset_name,
    yaml_config,
    data_root,
    log_path,
    ssb_mode="full_ssb",
    ssb_alpha=0.40,
    ssb_beta=0.20,
    ssb_tau_text=0.10,
    ssb_tau_visual=0.10,
    ssb_min_weight=1.00,
    ssb_max_weight=3.60,
    epochs=500,
    batch_size=None,
    lr=1e-3,
    weight_decay=None,
    seed=1,
):
    print("=" * 85)
    print(f"🚀 KHỞI CHẠY HUẤN LUYỆN STAIR-v5 (STAIR-BSC-Reweight): {dataset_name}")
    print(f"  * Config YAML   : {yaml_config}")
    print(f"  * Data Root     : {data_root}")
    print(f"  * Log File      : {log_path}")
    print(f"  * SSB Mode      : '{ssb_mode}' (alpha={ssb_alpha}, beta={ssb_beta})")
    print(f"  * Tau Thresh    : tau_t={ssb_tau_text}, tau_v={ssb_tau_visual} | Weight Clamp: [{ssb_min_weight}, {ssb_max_weight}]")
    print(f"  * Training Specs: Epochs={epochs}, LR={lr}, Seed={seed}")
    print("=" * 85)

    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    stop_evt = threading.Event()
    vram_thread = threading.Thread(target=vram_monitor, args=(dataset_name, stop_evt), daemon=True)
    vram_thread.start()

    t0 = time.time()
    cmd = [
        sys.executable, '-u', os.path.join(STAIR_DIR, 'mainS3_v5.py'),
        '--config', yaml_config,
        '--root', data_root,
        '--dataset', dataset_name,
        '--epochs', str(epochs),
        '--lr', str(lr),
        '--seed', str(seed),
        '--ssb-mode', str(ssb_mode),
        '--ssb-alpha', str(ssb_alpha),
        '--ssb-beta', str(ssb_beta),
        '--ssb-tau-text', str(ssb_tau_text),
        '--ssb-tau-visual', str(ssb_tau_visual),
        '--ssb-min-weight', str(ssb_min_weight),
        '--ssb-max-weight', str(ssb_max_weight),
    ]
    if batch_size is not None:
        cmd.extend(['--batch-size', str(batch_size)])
    if weight_decay is not None:
        cmd.extend(['--weight-decay', str(weight_decay)])
    if torch.cuda.is_available():
        cmd.extend(['--device', 'cuda:0'])

    proc_env = dict(os.environ, PYTHONUNBUFFERED='1')
    with open(log_path, 'w', encoding='utf-8') as logf:
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            cwd=STAIR_DIR,
            env=proc_env,
            text=True,
            bufsize=1,
            universal_newlines=True
        )
        for line in proc.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
            logf.write(line)
            logf.flush()
        proc.wait()
        logf.flush()

    stop_evt.set()
    vram_thread.join(timeout=3)
    elapsed = time.time() - t0

    print("\n" + "=" * 85)
    log_sz = os.path.getsize(log_path) if os.path.exists(log_path) else 0
    if proc.returncode == 0:
        print(f"✅ [HOÀN TẤT] Huấn luyện {dataset_name} thành công trong {elapsed/60:.1f} phút ({elapsed:.0f}s)!")
    else:
        print(f"❌ [THẤT BẠI] Huấn luyện kết thúc với mã lỗi: {proc.returncode}")

    best_ep, metrics = parse_best_metrics(log_path)
    print(f"  * Checkpoint tối ưu : Epoch {best_ep}")
    ref_data = BENCHMARK_TARGETS.get(dataset_name, {})
    base_m = ref_data.get('baseline', {})
    v4_1_m = ref_data.get('v4_1_ssb', {})

    print("\n--- BẢNG ĐỐI CHIẾU KẾT QUẢ VỚI BASELINE (03_stair.tex) ---")
    for m_name in ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']:
        val = metrics.get(m_name, None)
        if val is not None:
            b_val = base_m.get(m_name, 0.0)
            delta_b = ((val - b_val) / b_val * 100) if b_val > 0 else 0.0
            print(f"  {m_name:10s} : {val:.4f} | Baseline: {b_val:.4f} ({delta_b:+.2f}%)")
    print("=" * 85)
    return best_ep, metrics


## 🏃 4. THỰC NGHIỆM 1: AMAZON SPORTS (Điểm Bứt Phá Mục Tiêu)
Trên tập dữ liệu Amazon Sports, v4.1-SSB trước đó đã chứng minh tính hiệu quả (+0.45% Recall@20).
Ở phiên bản v5, với cơ chế tăng cường nhân Multiplicative Consensus Boost ($\alpha=0.40, \beta=0.20$) và SVD Whitening đẳng hướng chuẩn xác, mô hình hướng tới bứt phá vượt ngưỡng Baseline ($0.1111 \to 0.1120+$).


In [ ]:
# Cell 7: Huấn luyện STAIR-v5 trên Amazon Sports (full_ssb, alpha=0.40, beta=0.20)
SPORTS_CONFIG = os.path.join(STAIR_DIR, 'configs', 'Amazon2014Sports_550_MMRec.yaml')
SPORTS_LOG = os.path.join(LOG_DIR_V5, 'Amazon2014Sports_550_MMRec_full_ssb.log')

best_ep_sports, metrics_sports = run_training_v5(
    dataset_name='Amazon2014Sports_550_MMRec',
    yaml_config=SPORTS_CONFIG,
    data_root=DATA_ROOT,
    log_path=SPORTS_LOG,
    ssb_mode='full_ssb',
    ssb_alpha=0.40,
    ssb_beta=0.20,
    ssb_tau_text=0.10,
    ssb_tau_visual=0.10,
    ssb_min_weight=1.00,
    ssb_max_weight=3.60,
    epochs=500,
    lr=1e-3,
    seed=1
)


## 🏃 5. THỰC NGHIỆM 2: AMAZON BABY (Kiểm Định Phục Hồi Đồ Thị Siêu Thưa)
Trên tập dữ liệu Amazon Baby, đồ thị siêu thưa với tần suất đồng mua thấp khiến việc cộng dồn co-occurrence noise từng làm giảm hiệu năng ở v4.1 (-9.12%).
Ở v5, chế độ `modal_only` (hoặc `full_ssb` với $\beta=0.10$) được kích hoạt để đánh giá khả năng phục hồi về sát baseline ($0.1042$) và kiểm định giả thuyết trần tô-pô đồ thị tĩnh.


In [ ]:
# Cell 8: Huấn luyện STAIR-v5 trên Amazon Baby (modal_only hoặc full_ssb)
BABY_CONFIG = os.path.join(STAIR_DIR, 'configs', 'Amazon2014Baby_550_MMRec.yaml')
BABY_LOG = os.path.join(LOG_DIR_V5, 'Amazon2014Baby_550_MMRec_v5.log')

# Có thể lựa chọn ssb_mode='modal_only' (khuyến nghị cho Baby) hoặc 'full_ssb'
best_ep_baby, metrics_baby = run_training_v5(
    dataset_name='Amazon2014Baby_550_MMRec',
    yaml_config=BABY_CONFIG,
    data_root=DATA_ROOT,
    log_path=BABY_LOG,
    ssb_mode='modal_only',   # Đánh giá độc lập đồng thuận modal trên đồ thị siêu thưa
    ssb_alpha=0.40,
    ssb_beta=0.00,
    ssb_tau_text=0.10,
    ssb_tau_visual=0.10,
    ssb_min_weight=1.00,
    ssb_max_weight=3.60,
    epochs=500,
    lr=1e-3,
    seed=1
)


## 🏃 6. THỰC NGHIỆM 3: AMAZON ELECTRONICS (Quy Mô Lớn 63K Sản Phẩm)
Chỉ kích hoạt khi đã có sẵn dữ liệu `Amazon2014Electronics_550_MMRec` trong `/kaggle/input`.


In [ ]:
# Cell 9: Huấn luyện STAIR-v5 trên Amazon Electronics (nếu có dữ liệu)
ELEC_CONFIG = os.path.join(STAIR_DIR, 'configs', 'Amazon2014Electronics_550_MMRec.yaml')
ELEC_LOG = os.path.join(LOG_DIR_V5, 'Amazon2014Electronics_550_MMRec_full_ssb.log')

elec_dir = os.path.join(LOCAL_PROCESSED, 'Amazon2014Electronics_550_MMRec')
if os.path.exists(elec_dir) and len(os.listdir(elec_dir)) >= 5:
    print("🚀 Bắt đầu huấn luyện Amazon Electronics (Quy mô lớn 63K items)...")
    best_ep_elec, metrics_elec = run_training_v5(
        dataset_name='Amazon2014Electronics_550_MMRec',
        yaml_config=ELEC_CONFIG,
        data_root=DATA_ROOT,
        log_path=ELEC_LOG,
        ssb_mode='full_ssb',
        ssb_alpha=0.40,
        ssb_beta=0.20,
        epochs=500,
        batch_size=4096,
        lr=1e-3,
        seed=1
    )
else:
    print("ℹ️ Bỏ qua Amazon Electronics vì chưa phát hiện dữ liệu trong thư mục Processed/.")


In [ ]:
# Cell 10: Tổng Hợp Kết Quả & Xuất Bản Báo Cáo
from prettytable import PrettyTable

table = PrettyTable()
table.field_names = ["Tập Dữ Liệu", "Chế Độ v5", "Metric", "Baseline (03_stair.tex)", "STAIR-v5", "Độ Lệch (%)"]

results_map = [
    ('Amazon Sports', 'full_ssb', metrics_sports if 'metrics_sports' in locals() else {}, BENCHMARK_TARGETS['Amazon2014Sports_550_MMRec']['baseline']),
    ('Amazon Baby', 'modal_only', metrics_baby if 'metrics_baby' in locals() else {}, BENCHMARK_TARGETS['Amazon2014Baby_550_MMRec']['baseline']),
]

for ds_title, mode_str, measured, base_ref in results_map:
    for m in ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']:
        v_meas = measured.get(m, 0.0)
        v_base = base_ref.get(m, 0.0)
        delta = ((v_meas - v_base) / v_base * 100) if v_base > 0 else 0.0
        table.add_row([ds_title, mode_str, m, f"{v_base:.4f}", f"{v_meas:.4f}" if v_meas > 0 else "N/A", f"{delta:+.2f}%" if v_meas > 0 else "N/A"])

print("=" * 85)
print("📊 BẢNG TỔNG KẾT HIỆU NĂNG STAIR-v5 (STAIR-BSC-Reweight) VS BASELINE CHUẨN TẮC")
print("=" * 85)
print(table)

# Nén thư mục logs để tải về từ Kaggle Output
subprocess.run(['tar', '-czvf', '/kaggle/working/stair_v5_logs.tar.gz', '-C', '/kaggle/working', 'logs/STAIR-v5'], check=False)
print("\n✅ Đã lưu trữ log thành công tại: /kaggle/working/stair_v5_logs.tar.gz")
